### LLM 에게 도구를 붙여주기
- LLM 은 기본적으로 질문하면 답변하는 모델
- 우리가 만든 함수(도구) 를 골라 쓸수 있도록 해준다.
- AI 에이전트의 핵심 - RAG 도 마찬가지

- LLM 에게 이렇게 얘기를 하는 방식
    - 너에게 이런저런 도구가 있다.(날씨 조회,검색)
    - LLM 이 필요로 할때(내가 만약 날씨 관련 질문을 한다면 -> 날씨 조회 도구 필요한 때)
    - 알아서 그 도구를 선택해서 실행하는 방식 -> AI 에이전트
    - LLM 이 스스로 판단해서 도구를 쓰고 결과도 확인해서 최종 결과를 주는 방식
- 작동방식
    - 함수 실행은 우리가 하고, 그 결과를 다시 LLM 이 받아서 최종 자연스러운 결과를 우리에게 주는 방식    

#### 전체 흐름
1. LLM 에게 사용 가능한 tool 목록을 알려줍니다.
2. LLM 이 필요할 때 알아서 이 함수를 실행하라고 요청합니다.
3. 우리가 그 함수를 실행합니다.
4. 함수 실행 결과를 LLM 에게 돌려주면 LLM 이 최종 답을 만들어 반환합니다.

- LLM -> 판단
- 우리 -> 실행

In [ ]:
# 가장 간단한 대화 - USER 만 쓰
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()       # api 
client = OpenAI()

In [2]:
# 날씨 조회 - 사실은 웹검색
def get_weather(city):
    # 실제로는 날씨 웹 검색이 들어가야 하는 거지만 , 임시로 고정값
    return f"{city}의 날씨는 맑음이고, 기온이 50도"



print(get_weather("신대방"))


신대방의 날씨는 맑음이고, 기온이 50도


### 1. 도구를 LLM 에게 설명하기
- LLM 에게 우리가 만든 이 함수를 설명
    - 이름, 설명, 인자
    - tools 에 그 자세한 명세를 적어야 합니다.
    - 함수 이름은 영문으로, 설명은 LLM 이 이해할수 있도록 적는 것 

In [3]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
}]
print("도구 정의 완료")

도구 정의 완료


#### 2. LLM 이 도구 호출 결정권을 줍니다.

In [6]:
messages = [
    {'role':'user','content':'신대방 날씨 어때?'}
]

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    tools=tools,
    reasoning_effort="none",
    messages=messages
)

In [15]:
call = response.choices[0].message.tool_calls[0]
call

ChatCompletionMessageFunctionToolCall(id='call_JQBsZveX88Cg6L2eyx4YTSkX', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')

In [16]:
print(call.function.name)

get_weather


In [17]:
import json
args =  json.loads(call.function.arguments)
type(args)

dict

In [18]:
tool_result = get_weather(**args)
print(tool_result)

신대방의 날씨는 맑음이고, 기온이 50도


#### 4. 결과를 LLM 에게

In [20]:
messages.append(response.choices[0].message)
messages

[{'role': 'user', 'content': '신대방 날씨 어때?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_JQBsZveX88Cg6L2eyx4YTSkX', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')]),
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_JQBsZveX88Cg6L2eyx4YTSkX', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')])]

In [22]:
messages.append({'role':'tool','tool_call_id':call.id,'content':tool_result})   # 툴 사용한 결과
messages

[{'role': 'user', 'content': '신대방 날씨 어때?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_JQBsZveX88Cg6L2eyx4YTSkX', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')]),
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_JQBsZveX88Cg6L2eyx4YTSkX', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_JQBsZveX88Cg6L2eyx4YTSkX',
  'content': '신대방의 날씨는 맑음이고, 기온이 50도'},
 {'role': 'tool',
  'tool_call_id': 'call_JQBsZveX88Cg6L2eyx4YTSkX',
  'content': '신대방의 날씨는 맑음이고, 기온이 50도'}]

In [23]:

response1 = client.chat.completions.create(
    model="gpt-5.6-luna",
    tools=tools,
    reasoning_effort="none",
    messages=messages
)

BadRequestError: Error code: 400 - {'error': {'message': "An assistant message with 'tool_calls' must be followed by tool messages responding to each 'tool_call_id'. The following tool_call_ids did not have response messages: call_JQBsZveX88Cg6L2eyx4YTSkX", 'type': 'invalid_request_error', 'param': 'messages.[2].role', 'code': None}}

In [24]:
# 하나의 함수로 만들어서 사용
available_tools = {"get_weather": get_weather}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("부산 날씨 알려줘", tools))


현재 부산은 **맑고 기온은 50도**입니다.


### Agent = LLM + 도구
- 날씨 조회 도구
- 비서 에이전트로 만들고 싶다면 -> 필요한 기능은?


- 필요한 기능은?
    - 일정 관련 기능
        1. 일정 정보 등록 도구
        2. 일정 정보 조회 도구
        3. 일정 수정 도구
        4. 일정 삭제 도구
    - 일기장을 쓰는 기능
        1. 파일에 내용을 쓰는 도구
        2. 그림추가 도구


In [25]:
def recommend_cloth(data):
    return f"{data}의 스타일은 너무 멋져요. 80년대 스타일 같아요."

tools = [{
    "type": "function",
    "function": {
        "name": "recommend_cloth",
        "description": "옷에 대해서 물어봤을때 코멘트를 해준다.",
        "parameters": {
            "type": "object",
            "properties": {
                "data": {"type": "string", "description": "코멘트 받고 싶은 옷 이름"},
            },
            "required": ["data"],
        },
    },
},

{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
}
]




In [26]:
# 하나의 함수로 만들어서 사용
available_tools = {"get_weather": get_weather,"recommend_cloth":recommend_cloth}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("부산 날씨 알려줘", tools))

부산은 현재 **맑고, 기온은 50도**입니다.


In [27]:
print(chat_with_tools("이 가죽 자켓 어때?", tools))

사진이나 자켓의 색상·디자인을 알려주시면 더 정확히 봐드릴게요. 일반적으로 가죽 자켓은 활용도가 높고 멋스럽지만, 핏이 너무 크거나 어깨가 뜨면 둔해 보일 수 있어요. 바지와 신발은 깔끔한 기본 아이템으로 맞추면 실패가 적습니다.


In [29]:
def get_stock_price(stock):
    # 실제로는 날씨 웹 검색이 들어가야 하는 거지만 , 임시로 고정값
    return f"{stock} 주가는 횡보 하고 있네요"


tools = [{
    "type": "function",
    "function": {
        "name": "recommend_cloth",
        "description": "옷에 대해서 물어봤을때 코멘트를 해준다.",
        "parameters": {
            "type": "object",
            "properties": {
                "data": {"type": "string", "description": "코멘트 받고 싶은 옷 이름"},
            },
            "required": ["data"],
        },
    },
},


{
    "type": "function",
    "function": {
        "name": "get_stock_price",
        "description": "주가 정보를 알려줄때 ",
        "parameters": {
            "type": "object",
            "properties": {
                "stock": {"type": "string", "description": "알고싶은 주가 이름"},
            },
            "required": ["stock"],
        },
    },
},

{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
}
]

### 실습
- 주가 알려주기
- 내가 질문한 내용을 파일로 저장하기 -> with open(file.md, w....)
- 아까 질문한 파일을 조회하기 -> (codex와 같이)
- 그 외 자유롭게 하셔도 됩니다.

In [31]:
# 하나의 함수로 만들어서 사용
available_tools = {"get_weather": get_weather,"recommend_cloth":recommend_cloth,"get_stock_price":get_stock_price}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("삼전 주가 알려줘", tools))

삼성전자(삼전) 주가는 현재 **횡보세**를 보이고 있습니다.


In [35]:
def set_save_file(file_content):
    # 실제로는 날씨 웹 검색이 들어가야 하는 거지만 , 임시로 고정값
    with open("save_sample.txt", "w", encoding="utf-8") as file:
        file.write(file_content)


tools = [{
    "type": "function",
    "function": {
        "name": "recommend_cloth",
        "description": "옷에 대해서 물어봤을때 코멘트를 해준다.",
        "parameters": {
            "type": "object",
            "properties": {
                "data": {"type": "string", "description": "코멘트 받고 싶은 옷 이름"},
            },
            "required": ["data"],
        },
    },
},


{
    "type": "function",
    "function": {
        "name": "set_save_file",
        "description": "파일 로 내용 저장할때 사용하는 함수 ",
        "parameters": {
            "type": "object",
            "properties": {
                "file_content": {"type": "string", "description": "저장하고 싶은 내용"},
            },
            "required": ["file_content"],
        },
    },
},


{
    "type": "function",
    "function": {
        "name": "get_stock_price",
        "description": "주가 정보를 알려줄때 ",
        "parameters": {
            "type": "object",
            "properties": {
                "stock": {"type": "string", "description": "알고싶은 주가 이름"},
            },
            "required": ["stock"],
        },
    },
},


{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
}
]

In [36]:
# 하나의 함수로 만들어서 사용
available_tools = {"get_weather": get_weather,"recommend_cloth":recommend_cloth,"get_stock_price":get_stock_price,"set_save_file":set_save_file}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("이거 중요한 내용임 잠온다. 파일로 저장해줘", tools))

저장할 내용을 보내주세요. 내용이 있어야 파일로 저장할 수 있습니다.


In [48]:
response = client.responses.create(
    model="gpt-5.6-luna",
    tools=[{"type": "web_search"}],
    input="부산 날씨 어때?"
)

print(response.output[1].content[0].text)

AttributeError: 'ResponseFunctionWebSearch' object has no attribute 'content'

###  오픈 AI 에서 제공하는 툴
- 내장 : web_search, code_interpreter 코드가 잘 실행 되는지
- 직접 만든 함수 :